# K-Heroes 역사 인물 및 시나리오 데이터 정제 (character.json 생성)

1. **기본 데이터 로드**: `kf_area_total_merged.csv` 파일을 로드하고 결측치 처리.
2. **인물 필터링**: 이야기 건수가 **50건 이상**인 인물 34명 + 핵심 호국 인물인 **이순신**(40건)을 더해 총 35명의 가용 인물 후보군 설정.
3. **연관 스토리 매핑 (`associated_stories`)**: 인물별 관련 이야기를 찾아 **domain**('cltur'/'prsn')과 **id**('data_manage_no')를 분리 매핑하고 제목, 요약본 등을 객체로 수집.
4. **OpenAI 프로필 및 카테고리 생성**: OpenAI API를 연동해 인물별 RAG 스토리를 분석하고, **인물 분류 카테고리**('정치 / 외교', '독립 / 호국', '예술 / 문학', '실학 / 학문')를 동적으로 진단하며 MBTI, 스탯, 명언, 상황 설명 등 게임 카드를 위한 세부 데이터를 생성.
5. **파일 내보내기**: 최종 정제된 전체 사전을 `backend/data/characters.json`으로 저장.

In [ ]:
import os
import json
import time
import pandas as pd
import numpy as np
from collections import Counter
from dotenv import load_dotenv
from openai import OpenAI

# 환경 변수 (.env) 로드
BASE_DIR = ".."
dotenv_path = os.path.join(BASE_DIR, "..", ".env")
load_dotenv(dotenv_path)

print(f"Base Directory: {os.path.abspath(BASE_DIR)}")
print(f"Dotenv loaded from: {os.path.abspath(dotenv_path)}")

Base Directory: /Users/ahramkim/Documents/GitHub/k-heroes/backend
Dotenv loaded from: /Users/ahramkim/Documents/GitHub/k-heroes/.env


In [ ]:
# CSV 마스터 데이터 로드
CSV_PATH = os.path.join(BASE_DIR, "data", "proceed", "kf_area_total_merged.csv")
OUTPUT_JSON_PATH = os.path.join(BASE_DIR, "data", "characters.json")

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
    print(f"[SUCCESS] 데이터 로드 성공: {len(df):,}행, {df.shape[1]}개 열")
else:
    raise FileNotFoundError(f"[ERROR] 데이터를 찾을 수 없습니다: {CSV_PATH}")

[SUCCESS] 데이터 로드 성공: 26,876행, 22개 열


In [22]:
# relate_prsn_nm을 기준으로 인물별 이야기 수 집계
# 결측치 제거 및 공백 정리
df_clean = df.dropna(subset=["relate_prsn_nm"]).copy()
df_clean["relate_prsn_nm"] = df_clean["relate_prsn_nm"].astype(str).str.strip()
df_clean = df_clean[df_clean["relate_prsn_nm"] != ""]

# 인물별 관련 이야기 수 계산
person_counts = df_clean["relate_prsn_nm"].value_counts()

# 60개 이상인 인물 필터링 + 이순신 예외 포함
min_stories_threshold = 60
selected_characters = [name for name, count in person_counts.items() if count >= min_stories_threshold]

# 주요 영웅인 이순신 강제 포함 예외 처리
special_characters = ["이순신"]
for name in special_characters:
    if name not in selected_characters and name in person_counts:
        selected_characters.append(name)

selected_characters = sorted(selected_characters)

print(f"선택된 플레이어블 캐릭터 수: {len(selected_characters)}명")
print("\n--- 캐릭터별 이야기 수 ---")
for name in selected_characters:
    print(f"{name}: {person_counts[name]}건")

선택된 플레이어블 캐릭터 수: 31명

--- 캐릭터별 이야기 수 ---
고종: 150건
공자: 70건
광해군: 70건
김창환: 73건
김홍도: 64건
박두진: 65건
박목월: 66건
서정주: 75건
선조: 60건
송만갑: 100건
송시열: 83건
순종: 69건
영조: 66건
유치환: 72건
이광수: 78건
이동백: 71건
이순신: 39건
이승만: 78건
이이: 68건
이태준: 75건
이황: 78건
인조: 95건
정선: 64건
정약용: 62건
정정렬: 78건
정조: 114건
정철: 67건
최승희: 62건
한성준: 85건
허균: 80건
흥선대원군: 70건


In [ ]:
def get_associated_stories_for_char(character_name, max_stories=50):
    """
    CSV에서 특정 인물과 관련성이 높은 이야기(스토리) 목록을 추출하여
    프론트엔드 렌더링에 적합한 메타데이터 구조로 가공합니다.
    """
    # 인물 이름이 포함된 행 필터링
    char_rows = df_clean[df_clean["relate_prsn_nm"].astype(str).str.contains(character_name)].copy()
    
    # 중복 이야기 타이틀을 제거하기 위해 data_title_nm 기준으로 unique 화
    char_rows = char_rows.drop_duplicates(subset=["data_title_nm"]).head(max_stories)
    
    stories = []
    for _, row in char_rows.iterrows():
        domain = str(row["data_manage_keyword"]).strip()
        no = int(row["data_manage_no"])
        
        stories.append({
            "id": no,
            "domain": domain,
            "title": row["data_title_nm"],
            "summary": row["sumry_cn"] if pd.notna(row["sumry_cn"]) else "",
            "sido": row["ctprvn_nm"] if pd.notna(row["ctprvn_nm"]) else "",
            "sigungu": row["signgu_nm"] if pd.notna(row["signgu_nm"]) else ""
        })
    return stories

# 이순신 예시 출력
lee_stories = get_associated_stories_for_char("이순신")
print(f"이순신 매핑된 스토리 수: {len(lee_stories)}개")
print(json.dumps(lee_stories[:2], ensure_ascii=False, indent=2))

이순신 매핑된 스토리 수: 40개
[
  {
    "id": 8047,
    "domain": "prsn",
    "title": "유성룡의 종가, 안동하회마을 충효당",
    "summary": "충효당(忠孝堂)은 경상북도 안동시 풍천면 종가길 69(하회리)에 있는 조선시대의 가옥형 정자이다. 보물 제414호로 지정되어 있다. 충효당은 조선 중기의 문신 서애(西厓) 유성룡(柳成龍:1542∼1607)의 종택이다. 유성룡은 관직에서 물러나 고향으로 낙향하여 농환재(弄丸齋)라 불리는 풍산 서미동(西美洞)의 초가삼간에서 별세하였다. 이에 유성룡의 손자인 졸재(拙齋) 유원지(柳元之:1598～1674)가 유성룡의 학덕과 업적을 추모하기 위해 유림과 제자들의",
    "sido": "경상북도",
    "sigungu": "안동시"
  },
  {
    "id": 8048,
    "domain": "prsn",
    "title": "절의의 상징 거북바위 정자, 영양 삼귀정",
    "summary": "삼귀정은 경상북도 영양군 영양읍 대천리 945에 있는 조선시대의 정자이다. 경상북도시도유형문화재 제232호로 지정되어 있다. 용계(龍溪) 오흡(吳?:1576~1641)이 병자호란 때 남한산성이 함락되고 인조가 삼전도에서 항복하자 비분강개하여 세상과의 인연을 끊고 고향인 영양군 영양읍 대천리 반월산 아래에 초가 정자를 짓고 은거하였다. 이후 후손들이 기와집으로 개축하여 오늘에 이른다. 삼귀정이란 정자명은 삼귀정 앞에 정자를 등에 업은 듯한 형상의 세 거북",
    "sido": "경상북도",
    "sigungu": "영양군"
  }
]


In [24]:
# OpenAI API 클라이언트 생성
openai_api_key = os.environ.get("OPENAI_API_KEY")
if not openai_api_key:
    print("[WARNING] OPENAI_API_KEY 환경변수가 존재하지 않습니다. .env를 확인하십시오.")
    openai_client = None
else:
    openai_client = OpenAI(api_key=openai_api_key)
    print("[SUCCESS] OpenAI API 클라이언트 초기화 완료")

def generate_character_profile_via_openai(character_name, stories):
    """
    OpenAI GPT-4o-mini 모델을 사용하여 캐릭터의 역사적 행적 RAG 컨텍스트를 분석하고,
    게임 플레이를 위한 카테고리, MBTI와 캐릭터 스탯 등 상세 정보를 JSON 형식으로 생성합니다.
    """
    if not openai_client:
        raise ValueError("OpenAI API 클라이언트가 존재하지 않아 생성을 진행할 수 없습니다.")
        
    # RAG 컨텍스트 텍스트 작성
    context_str = ""
    for i, s in enumerate(stories[:15]): # API 컨텍스트 한계를 고려해 상위 15개 스토리 요약 활용
        context_str += f"[스토리 {i+1}] 제목: {s['title']}, 요약: {s['summary']}\n\n"

    prompt = f"""
너는 역사 선택형 시뮬레이션 게임 'K-Heroes'의 인물 카드 설계자야.
제공된 RAG 백그라운드 지식을 기반으로 대상 인물에 대한 정보를 추출하여 재미있는 인물 카드 데이터로 생성해 줘.
어려운 한자어는 피하고 초등학생도 쉽게 읽을 수 있는 단어를 써줘.

[대상 인물]
이름: {character_name}

[RAG 데이터 컨텍스트]
{context_str}

[캐릭터 분류 카테고리 판별 원칙 - 매우 중요!]
RAG 데이터 컨텍스트에 담긴 인물의 주된 업적과 행적을 분석하여 다음 4가지 테마 중 하나를 엄격히 부여하세요:
- 정치 / 외교: 왕, 대통령, 재상, 정치가로서 국가 제도 개혁, 외교 협상, 권력 투쟁 등을 주로 펼친 경우.
- 독립 / 호국: 국난 극복, 왜구 방어, 의병 활동, 독립운동, 군사적 전투 등을 주로 이끈 장군, 의사, 열사 등.
- 예술 / 문학: 판소리, 서양화, 풍속화, 무용, 시, 소설, 음악 등 문화예술 창작 활동을 한 예인, 작가, 화가 등.
- 실학 / 학문: 성리학, 실학, 고증학, 과학 기술 연구 및 학술 교육에 헌신한 학자, 사상가 등.

[캐릭터 설계 절대 원칙 - MBTI 자동 판별]
★ [매우 중요] 특정 MBTI를 미리 정해두고 인물의 행적을 억지로 짜맞추지 마세요. 반드시 제공된 [RAG 데이터 컨텍스트]를 먼저 철저히 분석한 뒤, 인물이 역사 속에서 보여준 가장 주된 업적과 성향에 가장 잘 어울리는 MBTI를 16가지 유형 중 하나로 '스스로 판별'하여 부여하세요.
역사적 사실을 다 집어넣으려고 한 문장 안에 반대되는 성향을 섞어 쓰는 순간 너의 임무는 실패입니다. 설정한 MBTI 알파벳 성향 하나에만 100% 집중하여 선명한 캐릭터 카드를 만드세요.

[MBTI 부여 및 작성 원칙 - 매우 중요!]
★ MBTI 알파벳별 역사적 행동 매칭 기준 (오류 방지 이분법 기준):
AI는 인물의 수많은 업적 중, 본인이 판별하여 선택한 알파벳의 '생각과 행동 목적'에 100% 부합하는 사례만 남기고 반대 성향의 사실은 반드시 삭제해야 합니다.

[E vs I : 에너지를 쓰고 결단을 내리는 방식]
- E (외향 - 외부 교류 중심): 여러 사람 앞에 나서서 대중을 이끌거나 외부에 에너지를 쏟은 행적입니다.
  * 키워드: 대중 연설, 적극적인 동료/군사 규합, 격렬한 끝장 토론, 활발한 대외 외교 활동.
- I (내향 - 내부 집중 중심): 남들의 시선에서 벗어나 혼자 사색하거나 소수의 측근과 은밀하게 움직인 행적입니다.
  * 키워드: 독자적인 책/일기 저술, 홀로 밤새 고민함, 밀실 정치, 비밀 편지(어찰)를 통한 고독한 결단.

[S vs N : 정보를 바라보고 목적을 세우는 방식] ★가장 오차가 크니 주의할 것★
- S (감각 - 현실과 과거 중심): "이미 증명된 과거의 기록, 선대의 관습, 눈앞의 실제 데이터와 현장 경험"을 기반으로 움직인 행적입니다.
  * 키워드: 과거 역사 기록 참고, 전통문화와 관습 수호, 지형과 날씨 데이터 활용, 실제 농사/실측 경험 중시.
- N (직관 - 미래와 혁신 중심): "당장 눈앞에 없는 미래의 가능성, 기존 역사를 깨부수는 독창적인 비전과 발명"을 기반으로 움직인 행적입니다.
  * 키워드: 세상에 없던 새로운 문자 창제(한글), 신분제 폐지 구상, 서양의 첨단 신기술/문물 전격 도입, 미래를 내다본 수도 천도.

[T vs F : 판단과 의사결정을 내리는 기준]
- T (사고 - 논리와 실리 중심): 개인적인 감정이나 도덕적 명분보다 "철저한 인과관계 분석과 실리적 이익"을 우선시한 행적입니다.
  * 키워드: 냉철한 정세/리스크 분석, 국가적 실리(이익) 저울질, 법과 원칙에 따른 엄격한 처벌과 집행.
- F (감정 - 인간미와 명분 중심): 이익 계산보다 "사람 중심의 가치, 백성에 대한 사랑, 도덕적 명분과 유대감"을 우선시한 행적입니다.
  * 키워드: 백성을 가엽게 여김(애민정신), 동료들과의 끈끈한 의리, 나라를 향한 뜨거운 충성심, 눈물의 호소.

[J vs P : 목표를 계획하고 실행하는 방식]
- J (판단 - 체계와 통제 중심): 변수를 줄이기 위해 "사전에 치밀한 계획을 세우고 체계적인 시스템"을 정비한 행적입니다.
  * 키워드: 치밀한 사전 계획 수립, 국가 제도 및 법전 완성, 매뉴얼화, 엄격한 군율과 규칙 확립.
- P (인식 - 유연과 기동 중심): 틀에 얽매이지 않고 "상황의 변화에 따라 자유롭고 임기응변식으로 대응"한 행적입니다.
  * 키워드: 예상치 못한 위기에서의 기발한 임기응변, 고정관념에서 벗어난 자유로운 탐색, 신출귀몰한 게릴라 전술(의병 활동).

반드시 아래 JSON 형식으로만 출력해:
{{
    "name": "인물 이름 (예: 고종, 이순신, 세종대왕)",
    "category": "RAG 컨텍스트를 분석하여 부여한 카테고리. 반드시 다음 4가지 문자열 중 하나여야 함: ['정치 / 외교', '독립 / 호국', '예술 / 문학', '실학 / 학문']",
    "era": "시대 명칭 (예: 조선 후기(1863-1907), 조선 시대(1545-1598))",
    "era_tag": "시대 태그 (예: 조선 후기, 조선 시대, 일제강점기, 근대)",
    "role": "대표 직업/역할 (예: 왕, 독립운동가, 서양화가)",
    "keywords": ["대표 해시태그 1", "태그 2", "태그 3"],
    "years": "생몰년도 또는 활동 기간 (예: 1863-1907, 1545-1598)",
    "situation": "당시 시대 상황 설명 (쉽게 2~3줄)",
    "one_line_summary": "히어로물 느낌의 직관적인 수식어 한줄 요약",
    "mbti": "RAG 분석을 통해 인물에게 가장 잘 어울린다고 판별한 MBTI 4글자",
    "mbti_nickname": "부여한 MBTI에 따른 캐릭터 별명",
    "mbti_details": {{
        "E_I": "최종 선택한 글자가 'E'라면 [활발한 대외 활동/토론] 행적을 쓰고, 'I'라면 [혼자 사색/독자적 저술/고독한 결단] 행적만 쓰세요. (두 성향 절대 섞지 말 것)",
        "S_N": "최종 선택한 글자가 'S'라면 [전통 수호/기존 기록 참고]를 적고, 'N'이라면 [첨단 기술 도입/새로운 비전 창조] 행적만 적으세요. (두 성향 절대 섞지 말 것)",
        "T_F": "최종 선택한 글자가 'T'라면 [냉철한 인과 분석/실리 계산] 행적을 쓰고, 'F'라면 [백성을 향한 사랑(애민)/의리와 인간미] 행적만 쓰세요. (두 성향 절대 섞지 말 것)",
        "J_P": "최종 선택한 글자가 'J'라면 [치밀한 사전 계획/체계적 시스템 정비] 행적을 쓰고, 'P'라면 [유연한 임기응변/기동성 있는 전술] 행적만 쓰세요. (두 성향 절대 섞지 말 것)"
    }},
    "stats": [
        {{"name": "스탯 1 이름 (예: 외교력)", "value": 88, "desc": "설명"}},
        {{"name": "스탯 2 이름 (예: 개혁 추진력)", "value": 90, "desc": "설명"}},
        {{"name": "스탯 3 이름 (예: 애민정신)", "value": 85, "desc": "설명"}}
    ],
    "intro_quote": "인물의 유명한 명언이나 다짐 한 줄",
    "intro_desc": "인물이 활약하게 된 주요 계기와 행적 설명 2~3줄"
}}
"""
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"}
    )
    
    card_data = json.loads(response.choices[0].message.content)
    return card_data

[SUCCESS] OpenAI API 클라이언트 초기화 완료


In [25]:
character_database = {} 

test_selected_characters = selected_characters[:2]
print(f"총 {len(test_selected_characters)}명의 인물 카드 생성을 시작합니다. (테스트 모드: {test_selected_characters})")
start_time = time.time()

for idx, char_name in enumerate(test_selected_characters):
    print(f"\n[{idx+1}/{len(test_selected_characters)}] {char_name} 처리 중...")
    
    # 1. 시나리오 데이터 맵핑
    associated_stories = get_associated_stories_for_char(char_name)
    
    # 2. OpenAI 기반 인물 프로필 정보 생성 (동적 카테고리 판별 및 MBTI 도출)
    retries = 3
    for attempt in range(retries):
        try:
            profile_data = generate_character_profile_via_openai(char_name, associated_stories)
            
            # 연관 시나리오 데이터 보강
            profile_data["associated_stories"] = associated_stories
            
            character_database[char_name] = profile_data
            category = profile_data.get("category", "예술 / 문학")
            print(f" ➔ {char_name} 프로필 생성 완료! (카테고리: {category}, MBTI: {profile_data.get('mbti')})")
            break
        except Exception as e:
            print(f" ➔ {char_name} 생성 시도 {attempt+1} 실패: {e}")
            if attempt < retries - 1:
                time.sleep(10 * (attempt + 1))
            else:
                print(f"[CRITICAL] {char_name} 생성 최종 실패. 오류 스킵합니다.")

    # API rate limit 방지용 딜레이
    time.sleep(2)

end_time = time.time()
print(f"\n전체 데이터 생성 완료! 소요 시간: {end_time - start_time:.2f}초")

총 2명의 인물 카드 생성을 시작합니다. (테스트 모드: ['고종', '공자'])

[1/2] 고종 처리 중...
 ➔ 고종 프로필 생성 완료! (카테고리: 정치 / 외교, MBTI: ENFJ)

[2/2] 공자 처리 중...
 ➔ 공자 프로필 생성 완료! (카테고리: 실학 / 학문, MBTI: NFJ)

전체 데이터 생성 완료! 소요 시간: 25.49초


In [26]:
# 카테고리 분포 요약 출력
categorized_dict = {}
for name, data in character_database.items():
    cat = data.get("category", "미지정")
    categorized_dict[cat] = categorized_dict.get(cat, []) + [name]

print("--- LLM 판별 결과 테마별 인물 분포 ---")
for cat, names in categorized_dict.items():
    print(f"[{cat}] ({len(names)}명): {', '.join(names)}")

# character.json으로 파일 저장
os.makedirs(os.path.dirname(OUTPUT_JSON_PATH), exist_ok=True)

with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(character_database, f, ensure_ascii=False, indent=4)

print(f"\n[SUCCESS] {len(character_database)}명의 캐릭터 데이터베이스가 성공적으로 저장되었습니다.")
print(f"출력 파일 경로: {OUTPUT_JSON_PATH}")

--- LLM 판별 결과 테마별 인물 분포 ---
[정치 / 외교] (1명): 고종
[실학 / 학문] (1명): 공자

[SUCCESS] 2명의 캐릭터 데이터베이스가 성공적으로 저장되었습니다.
출력 파일 경로: ../data/character.json
